## 🎯 Learning Objectives
* Demonstrate a comprehensive understanding of LangChain's core components for building AI agents.
* Apply advanced LangChain concepts, including tool integration, state management, and multi-agent coordination, to solve complex problems.
* Design and implement robust agentic workflows using LangGraph for sophisticated orchestration.
* Evaluate and debug multi-agent systems, ensuring reliable and performant operation.
* Synthesize knowledge from AG-03 and LLM-02 to build production-ready agent solutions.


# Final Assessment: Building AI Agents with LangChain (AG04-FA)

Welcome to the final assessment for AG-04, "Building AI Agents with LangChain"! This assessment is designed to evaluate your mastery of the concepts and practical skills acquired throughout the course. You've learned to construct sophisticated AI agents capable of using tools, managing state, and coordinating with other agents to achieve complex goals.

In the rapidly evolving landscape of AI in 2026, the ability to build robust, scalable, and intelligent agent systems is paramount. This course has equipped you with the knowledge to leverage LangChain's powerful abstractions, including the `Runnable` interface, LangGraph for stateful orchestration, and advanced tool integration techniques, to create production-grade agentic applications.

This assessment comprises two main parts:

1.  **Review Questions**: A set of theoretical questions to test your understanding of core concepts, design patterns, and best practices in agent development.
2.  **Capstone Coding Problem**: A practical, hands-on challenge requiring you to design and implement a multi-agent system using LangChain and LangGraph. This problem will test your ability to integrate various components, manage agent state, orchestrate complex workflows, and handle real-world scenarios.

Your success in this assessment will demonstrate your readiness to tackle advanced agentic AI projects and contribute significantly to the field of AI automation. Good luck!


## Review Questions

Answer the following questions concisely, demonstrating your understanding of the concepts covered in AG-04.

1.  Explain the primary advantages of using LangGraph over a traditional `AgentExecutor` for building complex, stateful agentic workflows. Provide an example scenario where LangGraph's benefits are particularly evident.
2.  Describe how custom tools are integrated into LangChain agents. What considerations are important when designing a custom tool for an agent, especially regarding its `name`, `description`, and `args_schema`?
3.  Discuss at least three different strategies for managing memory or state in a LangChain agent. When would you choose one strategy over another?
4.  How does the `Runnable` interface enhance the modularity and composability of LangChain components? Provide an example of chaining different `Runnable` objects to create a custom agentic step.
5.  Outline the key steps involved in designing a multi-agent system using LangGraph. What role does the `StateGraph` play, and how are conditional edges utilized for dynamic routing?
6.  What are some common challenges encountered when deploying LangChain agents to production environments in 2026? How can observability and error handling be effectively implemented in such systems?
7.  Compare and contrast the `AgentExecutor` with `RunnableAgent`. In what situations would you prefer one over the other for a single-agent task?
8.  Explain the concept of 'tool hallucination' in AI agents. What techniques can be employed to mitigate this issue?


## Capstone Coding Problem: Automated Content Generation Pipeline

**Scenario:**

Your task is to build an automated content generation pipeline using LangChain and LangGraph. The system should be able to take a user-provided topic, research it, synthesize the information, draft an article, and then review and refine it until it meets a satisfactory quality standard. This mimics a common workflow in digital publishing or marketing agencies.

**Requirements:**

1.  **Multi-Agent System**: Design a system with at least four distinct agents, each with a specific role:
    *   **Topic Research Agent**: Responsible for searching the web for information on the given topic.
    *   **Information Synthesis Agent**: Responsible for taking raw research results and creating a structured outline or key points.
    *   **Content Drafting Agent**: Responsible for writing a preliminary article based on the synthesized outline.
    *   **Review & Refinement Agent**: Responsible for evaluating the drafted article, providing feedback, and deciding if a re-draft is needed or if the article is complete.

2.  **Tool Use**: Integrate at least two different tools:
    *   A web search tool (e.g., `TavilySearchResults` or a similar modern search API).
    *   A custom tool for simulating 'saving' the final article (e.g., a simple Python function that prints the article to console or saves to a dummy file).

3.  **State Management**: Utilize LangGraph's `StateGraph` to manage the shared state across agents. The state should include:
    *   The initial `topic`.
    *   `raw_research_results` (from the research agent).
    *   `synthesized_outline` (from the synthesis agent).
    *   `current_draft` (from the drafting agent).
    *   `review_feedback` (from the review agent).
    *   A `status` field (e.g., 'researching', 'synthesizing', 'drafting', 'reviewing', 'revising', 'completed').
    *   A `revision_count` to track re-drafts.

4.  **Dynamic Workflow with LangGraph**: Implement the workflow using LangGraph, including:
    *   Sequential execution of agents.
    *   Conditional routing based on the `Review & Refinement Agent`'s decision (e.g., if feedback suggests re-drafting, loop back to the `Content Drafting Agent`; otherwise, proceed to completion).
    *   A maximum number of revisions to prevent infinite loops.

5.  **LLM Integration**: Use a modern LLM (e.g., OpenAI's latest models, Anthropic's Claude, or a local open-source model via Ollama/LiteLLM) for agent reasoning and content generation.

6.  **Robustness**: Include basic error handling (e.g., if a search fails, or if the revision count exceeds a limit).

**Output**: The final output should be a well-structured article on the given topic, along with a log of the agent's decision-making process.

**Example Flow:**

`Start -> Research Agent -> Synthesis Agent -> Drafting Agent -> Review Agent (if needs revision) -> Drafting Agent -> Review Agent (if complete) -> Save Article Tool -> End`


In [ ]:
# Capstone Problem: Automated Content Generation Pipeline - Starter Template

import os
from typing import List, Dict, TypedDict, Union

# Ensure you have these installed: pip install langchain langchain-openai langgraph tavily-python
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_community.tools import TavilySearchResults

# --- Configuration --- #
# Set your API keys here or as environment variables
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# --- Define Graph State --- #
class AgentState(TypedDict):
    topic: str
    raw_research_results: List[str]
    synthesized_outline: str
    current_draft: str
    review_feedback: str
    status: str # e.g., 'researching', 'synthesizing', 'drafting', 'reviewing', 'revising', 'completed'
    revision_count: int
    messages: List[BaseMessage]

# --- Define Tools --- #
# Web Search Tool
tavily_search = TavilySearchResults(max_results=5)

# Custom Tool: Save Article
@tool
def save_article(article_content: str, topic: str) -> str:
    """Saves the final article content to a file or prints it."""
    filename = f"final_article_{topic.replace(' ', '_').lower()}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(article_content)
    print(f"\n--- Article Saved to {filename} ---\n")
    print(article_content)
    return f"Article successfully saved to {filename}."

# List of all tools available to agents
# Note: Not all agents will use all tools. This is for the overall graph.
all_tools = [tavily_search, save_article]

# --- Define Agent Nodes (Placeholders) --- #

# Helper function to create an agent runnable
def create_agent_runnable(llm, tools, system_prompt):
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="messages"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    # This is a simplified agent. For full tool use, you'd integrate with AgentExecutor or similar.
    # For this template, we'll focus on direct LLM calls within the graph nodes.
    return prompt | llm.bind_tools(tools)

# Node for Topic Research Agent
def research_node(state: AgentState) -> AgentState:
    print("\n--- Research Agent: Starting ---")
    topic = state["topic"]
    # TODO: Implement research logic using tavily_search tool
    # Example: research_results = tavily_search.invoke({"query": topic})
    # Update state with raw_research_results and status
    print("Research Agent: Placeholder for research results.")
    return {
        "raw_research_results": ["Placeholder research result 1", "Placeholder research result 2"],
        "status": "synthesizing",
        "messages": state["messages"] + [HumanMessage(content="Research completed.")]
    }

# Node for Information Synthesis Agent
def synthesis_node(state: AgentState) -> AgentState:
    print("\n--- Synthesis Agent: Starting ---")
    raw_research = state["raw_research_results"]
    topic = state["topic"]
    # TODO: Implement synthesis logic to create an outline from raw_research
    # Example: outline = llm.invoke(f"Given this research: {raw_research}, create an outline for an article on {topic}.").content
    print("Synthesis Agent: Placeholder for synthesized outline.")
    return {
        "synthesized_outline": f"1. Introduction to {topic}\n2. Key aspects\n3. Conclusion",
        "status": "drafting",
        "messages": state["messages"] + [HumanMessage(content="Outline synthesized.")]
    }

# Node for Content Drafting Agent
def drafting_node(state: AgentState) -> AgentState:
    print("\n--- Drafting Agent: Starting ---")
    outline = state["synthesized_outline"]
    topic = state["topic"]
    # TODO: Implement drafting logic to create an article from the outline
    # Example: draft = llm.invoke(f"Write an article based on this outline: {outline} for topic {topic}.").content
    print("Drafting Agent: Placeholder for article draft.")
    return {
        "current_draft": f"This is a draft article about {topic}. It follows the outline: {outline}.",
        "status": "reviewing",
        "revision_count": state.get("revision_count", 0) + 1,
        "messages": state["messages"] + [HumanMessage(content="Article drafted.")]
    }

# Node for Review & Refinement Agent
def review_node(state: AgentState) -> AgentState:
    print("\n--- Review Agent: Starting ---")
    current_draft = state["current_draft"]
    topic = state["topic"]
    revision_count = state.get("revision_count", 0)
    max_revisions = 3 # Define a max revision limit

    # TODO: Implement review logic. Decide if the draft is good or needs revision.
    # Example: feedback = llm.invoke(f"Review this article draft: {current_draft} on {topic}. Provide feedback or state 'APPROVED'.").content
    # Based on feedback, set review_feedback and update status.

    if revision_count >= max_revisions:
        print(f"Review Agent: Max revisions ({max_revisions}) reached. Approving current draft.")
        return {
            "review_feedback": "Max revisions reached. Approving current draft.",
            "status": "completed",
            "messages": state["messages"] + [HumanMessage(content="Max revisions reached. Article approved.")]
        }

    # For template, let's simulate needing revision once, then approving
    if revision_count == 1:
        print("Review Agent: Draft needs revision. Simulating feedback.")
        return {
            "review_feedback": "The introduction needs more detail and examples. Expand on point 2.",
            "status": "revising",
            "messages": state["messages"] + [HumanMessage(content="Draft needs revision.")]
        }
    else:
        print("Review Agent: Draft approved.")
        return {
            "review_feedback": "Article is well-written and complete. Approved.",
            "status": "completed",
            "messages": state["messages"] + [HumanMessage(content="Article approved.")]
        }

# Node for Saving the Article
def save_node(state: AgentState) -> AgentState:
    print("\n--- Save Article Node: Starting ---")
    final_article = state["current_draft"]
    topic = state["topic"]
    # TODO: Call the save_article tool
    # Example: save_article.invoke({"article_content": final_article, "topic": topic})
    print(f"Save Article Node: Article for '{topic}' would be saved here.")
    return {
        "status": "completed",
        "messages": state["messages"] + [HumanMessage(content="Article saved.")]
    }

# --- Define Conditional Edges --- #
def decide_to_revise(state: AgentState) -> str:
    print(f"\n--- Deciding next step based on status: {state['status']} ---")
    if state["status"] == "revising":
        return "drafting_agent"
    elif state["status"] == "completed":
        return "save_article_node"
    else:
        # This should ideally not happen if statuses are managed correctly
        return "error_state" # Or raise an error

# --- Build the LangGraph Workflow --- #
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("research_agent", research_node)
workflow.add_node("synthesis_agent", synthesis_node)
workflow.add_node("drafting_agent", drafting_node)
workflow.add_node("review_agent", review_node)
workflow.add_node("save_article_node", save_node)

# Set entry point
workflow.set_entry_point("research_agent")

# Add edges
workflow.add_edge("research_agent", "synthesis_agent")
workflow.add_edge("synthesis_agent", "drafting_agent")
workflow.add_edge("drafting_agent", "review_agent")

# Conditional edge from review_agent
workflow.add_conditional_edges(
    "review_agent",
    decide_to_revise,
    {
        "drafting_agent": "drafting_agent", # Loop back for revision
        "save_article_node": "save_article_node" # Proceed to save
    }
)

workflow.add_edge("save_article_node", END)

# Compile the graph
app = workflow.compile()

# --- Run the Pipeline --- #
if __name__ == "__main__":
    initial_state = {
        "topic": "The Future of Quantum Computing in 2030",
        "raw_research_results": [],
        "synthesized_outline": "",
        "current_draft": "",
        "review_feedback": "",
        "status": "researching",
        "revision_count": 0,
        "messages": [HumanMessage(content="Starting content generation for topic: The Future of Quantum Computing in 2030")]
    }

    print("\n--- Starting Content Generation Pipeline ---")
    final_state = None
    for s in app.stream(initial_state):
        print(f"Current state: {s}")
        final_state = s

    print("\n--- Pipeline Completed ---")
    if final_state and final_state.get("save_article_node"):
        print(f"Final Article Status: {final_state['save_article_node']['status']}")
        print(f"Final Article Content (truncated): {final_state['save_article_node']['current_draft'][:200]}...")
    elif final_state and final_state.get("review_agent"):
        print(f"Final Article Status: {final_state['review_agent']['status']}")
        print(f"Final Article Content (truncated): {final_state['review_agent']['current_draft'][:200]}...")
    else:
        print("Pipeline did not reach a final state as expected.")

    # To visualize the graph (requires graphviz: pip install pygraphviz pydotplus)
    # from IPython.display import Image, display
    # display(Image(app.get_graph().draw_png()))


In [ ]:
# Capstone Problem: Automated Content Generation Pipeline - Detailed Solution

import os
from typing import List, Dict, TypedDict, Union

# Ensure you have these installed: pip install langchain langchain-openai langgraph tavily-python
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_community.tools import TavilySearchResults

# --- Configuration --- #
# Set your API keys here or as environment variables
# For production, use environment variables or a secure secret management system.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"

# Initialize LLM
# Using a capable model for complex reasoning and generation
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, max_tokens=1024)

# --- Define Graph State --- #
# The state object that will be passed between nodes in the LangGraph workflow.
class AgentState(TypedDict):
    topic: str # The initial topic for content generation
    raw_research_results: List[str] # Raw search results from the research agent
    synthesized_outline: str # Structured outline created by the synthesis agent
    current_draft: str # The current version of the article draft
    review_feedback: str # Feedback from the review agent
    status: str # Current status of the pipeline (e.g., 'researching', 'completed')
    revision_count: int # Tracks how many times the article has been revised
    messages: List[BaseMessage] # A log of messages for debugging and context

# --- Define Tools --- #
# Web Search Tool: Tavily for up-to-date web search capabilities.
# In 2026, we assume robust and fast search APIs are standard.
tavily_search = TavilySearchResults(max_results=5)

# Custom Tool: Save Article
@tool
def save_article(article_content: str, topic: str) -> str:
    """Saves the final article content to a file and prints it to the console.
    This simulates publishing or archiving the content."""
    filename = f"final_article_{topic.replace(' ', '_').lower()}.txt"
    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(article_content)
        print(f"\n--- Article Successfully Saved to {filename} ---\n")
        print(article_content)
        return f"Article successfully saved to {filename}."
    except Exception as e:
        print(f"Error saving article: {e}")
        return f"Failed to save article: {e}"

# List of all tools available to agents (though agents will only use relevant ones)
all_tools = [tavily_search, save_article]

# --- Define Agent Nodes --- #

# Helper function to create an LLM-based agent runnable
def create_agent_runnable(llm, tools, system_prompt):
    """Creates a LangChain runnable for an agent that can use tools."""
    # This uses the LangChain expression language (LCEL) for a flexible agent structure.
    # The agent will decide whether to call a tool or respond directly.
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="messages"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])
    return prompt | llm.bind_tools(tools)

# Node for Topic Research Agent
def research_node(state: AgentState) -> AgentState:
    print("\n--- Research Agent: Starting ---")
    topic = state["topic"]
    messages = state["messages"]

    research_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a diligent research assistant. Your goal is to find comprehensive and up-to-date information on the given topic using web search tools. Summarize your findings concisely."),
        HumanMessage(content=f"Research the following topic: {topic}. Provide key facts and relevant links."),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ])

    # Create a simple agent for research that uses Tavily
    research_agent_runnable = research_prompt | llm.bind_tools([tavily_search])

    # Invoke the search tool directly or through a simple agent logic
    try:
        print(f"Research Agent: Searching for '{topic}'...")
        search_results = tavily_search.invoke({"query": topic})
        raw_research = [str(search_results)] # Convert tool output to string list
        print("Research Agent: Search completed.")
        new_messages = messages + [HumanMessage(content=f"Research results for '{topic}': {raw_research}")]
    except Exception as e:
        print(f"Research Agent: Error during search: {e}")
        raw_research = [f"Error during research: {e}"]
        new_messages = messages + [HumanMessage(content=f"Research failed for '{topic}'.")]

    return {
        "raw_research_results": raw_research,
        "status": "synthesizing",
        "messages": new_messages
    }

# Node for Information Synthesis Agent
def synthesis_node(state: AgentState) -> AgentState:
    print("\n--- Synthesis Agent: Starting ---")
    raw_research = state["raw_research_results"]
    topic = state["topic"]
    messages = state["messages"]

    synthesis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert content strategist. Your task is to synthesize raw research into a structured, logical outline for an article. Focus on key points, sub-sections, and a clear flow."),
        HumanMessage(content=f"Given the following research results for the topic '{topic}':\n\n{raw_research}\n\nCreate a detailed article outline. Include an introduction, 3-5 main sections with sub-points, and a conclusion."),
    ])

    try:
        print("Synthesis Agent: Generating outline...")
        outline_response = llm.invoke(synthesis_prompt.format_messages(messages=messages)).content
        print("Synthesis Agent: Outline generated.")
        new_messages = messages + [HumanMessage(content=f"Synthesized outline: {outline_response}")]
    except Exception as e:
        print(f"Synthesis Agent: Error during synthesis: {e}")
        outline_response = f"Error during synthesis: {e}"
        new_messages = messages + [HumanMessage(content="Outline synthesis failed.")]

    return {
        "synthesized_outline": outline_response,
        "status": "drafting",
        "messages": new_messages
    }

# Node for Content Drafting Agent
def drafting_node(state: AgentState) -> AgentState:
    print("\n--- Drafting Agent: Starting ---")
    outline = state["synthesized_outline"]
    topic = state["topic"]
    messages = state["messages"]
    revision_count = state.get("revision_count", 0)

    drafting_prompt = ChatPromptTemplate.from_messages([
        ("system", f"You are a professional content writer. Your goal is to write a comprehensive and engaging article based on the provided outline and topic. Current revision count: {revision_count}. Incorporate any previous feedback if available."),
        HumanMessage(content=f"Write an article on the topic '{topic}' using the following outline:\n\n{outline}\n\nPrevious feedback (if any): {state['review_feedback'] or 'None'}\n\nEnsure the article is well-structured, informative, and engaging."),
    ])

    try:
        print(f"Drafting Agent: Generating draft (Revision {revision_count + 1})...")
        draft_response = llm.invoke(drafting_prompt.format_messages(messages=messages)).content
        print("Drafting Agent: Draft generated.")
        new_messages = messages + [HumanMessage(content=f"Article draft (Revision {revision_count + 1}): {draft_response}")]
    except Exception as e:
        print(f"Drafting Agent: Error during drafting: {e}")
        draft_response = f"Error during drafting: {e}"
        new_messages = messages + [HumanMessage(content="Article drafting failed.")]

    return {
        "current_draft": draft_response,
        "status": "reviewing",
        "revision_count": revision_count + 1,
        "messages": new_messages
    }

# Node for Review & Refinement Agent
def review_node(state: AgentState) -> AgentState:
    print("\n--- Review Agent: Starting ---")
    current_draft = state["current_draft"]
    topic = state["topic"]
    revision_count = state["revision_count"]
    messages = state["messages"]
    max_revisions = 3 # Define a maximum number of revisions to prevent infinite loops

    review_prompt = ChatPromptTemplate.from_messages([
        ("system", f"You are a critical content editor. Review the article draft for clarity, accuracy, completeness, and adherence to the topic '{topic}'. Provide constructive feedback if revision is needed, or state 'APPROVED' if the article is ready. Current revision: {revision_count}/{max_revisions}."),
        HumanMessage(content=f"Review the following article draft:\n\n{current_draft}\n\nIs it ready for publication? If not, provide specific feedback for improvement. If it is, simply respond with 'APPROVED'."),
    ])

    try:
        print("Review Agent: Reviewing draft...")
        review_response = llm.invoke(review_prompt.format_messages(messages=messages)).content
        print(f"Review Agent: Feedback: {review_response}")
        new_messages = messages + [HumanMessage(content=f"Review feedback: {review_response}")]

        if "APPROVED" in review_response.upper() and revision_count <= max_revisions:
            return {
                "review_feedback": review_response,
                "status": "completed",
                "messages": new_messages
            }
        elif revision_count >= max_revisions:
            print(f"Review Agent: Max revisions ({max_revisions}) reached. Forcing approval.")
            return {
                "review_feedback": f"Max revisions ({max_revisions}) reached. Approving current draft despite remaining feedback.",
                "status": "completed",
                "messages": new_messages
            }
        else:
            return {
                "review_feedback": review_response,
                "status": "revising",
                "messages": new_messages
            }
    except Exception as e:
        print(f"Review Agent: Error during review: {e}")
        return {
            "review_feedback": f"Error during review: {e}",
            "status": "completed", # Force completion on error to avoid loop
            "messages": messages + [HumanMessage(content="Review failed, forcing completion.")]
        }

# Node for Saving the Article
def save_node(state: AgentState) -> AgentState:
    print("\n--- Save Article Node: Starting ---")
    final_article = state["current_draft"]
    topic = state["topic"]
    messages = state["messages"]

    try:
        save_result = save_article.invoke({"article_content": final_article, "topic": topic})
        print(f"Save Article Node: {save_result}")
        new_messages = messages + [HumanMessage(content=f"Article saved: {save_result}")]
    except Exception as e:
        print(f"Save Article Node: Error saving article: {e}")
        new_messages = messages + [HumanMessage(content=f"Failed to save article: {e}")]

    return {
        "status": "completed",
        "messages": new_messages
    }

# --- Define Conditional Edges --- #
# This function determines the next node based on the 'status' field in the state.
def decide_to_revise(state: AgentState) -> str:
    print(f"\n--- Deciding next step based on status: {state['status']} ---")
    if state["status"] == "revising":
        return "drafting_agent" # Loop back to drafting for revisions
    elif state["status"] == "completed":
        return "save_article_node" # Article is approved, proceed to save
    else:
        # This case should ideally not be reached if statuses are managed correctly.
        # For robustness, we can log an error or raise an exception.
        print(f"ERROR: Unexpected status '{state['status']}' in decide_to_revise. Forcing completion.")
        return "save_article_node" # Fallback to save to prevent infinite loop

# --- Build the LangGraph Workflow --- #
workflow = StateGraph(AgentState)

# Add nodes to the graph, each representing an agent's task
workflow.add_node("research_agent", research_node)
workflow.add_node("synthesis_agent", synthesis_node)
workflow.add_node("drafting_agent", drafting_node)
workflow.add_node("review_agent", review_node)
workflow.add_node("save_article_node", save_node)

# Set the entry point for the graph (where the workflow begins)
workflow.set_entry_point("research_agent")

# Define the sequential flow between agents
workflow.add_edge("research_agent", "synthesis_agent")
workflow.add_edge("synthesis_agent", "drafting_agent")

# Define the conditional edge from the review agent
# This is where the dynamic decision-making happens: revise or complete.
workflow.add_conditional_edges(
    "review_agent",
    decide_to_revise, # The function that determines the next node
    {
        "drafting_agent": "drafting_agent", # If 'revising', go back to drafting
        "save_article_node": "save_article_node" # If 'completed', go to save
    }
)

# The final step: after saving, the workflow ends.
workflow.add_edge("save_article_node", END)

# Compile the graph into a runnable application
app = workflow.compile()

# --- Run the Pipeline --- #
if __name__ == "__main__":
    # Initial state for the content generation process
    initial_state = {
        "topic": "The Impact of AI on Creative Industries by 2030",
        "raw_research_results": [],
        "synthesized_outline": "",
        "current_draft": "",
        "review_feedback": "",
        "status": "researching",
        "revision_count": 0,
        "messages": [HumanMessage(content="Starting content generation for topic: The Impact of AI on Creative Industries by 2030")]
    }

    print("\n--- Starting Content Generation Pipeline ---")
    final_state = None
    # Stream the execution to see the state changes at each step
    for s in app.stream(initial_state):
        # The 's' variable contains the state updates from the last node executed.
        # We merge it into final_state to get the complete picture at the end.
        if isinstance(s, dict):
            if final_state is None:
                final_state = s
            else:
                final_state.update(s)
        print(f"Current state after node execution: {s}")

    print("\n--- Pipeline Completed ---")
    if final_state:
        print(f"Final Article Status: {final_state.get('status', 'Unknown')}")
        print(f"Final Revision Count: {final_state.get('revision_count', 0)}")
        print(f"Final Article Content (first 500 chars):\n{final_state.get('current_draft', 'No draft generated.')[:500]}...")
        print(f"\nFull final state: {final_state}")
    else:
        print("Pipeline did not produce a final state.")

    # To visualize the graph (requires graphviz: pip install pygraphviz pydotplus)
    # from IPython.display import Image, display
    # try:
    #     display(Image(app.get_graph().draw_png()))
    # except Exception as e:
    #     print(f"Could not draw graph. Ensure graphviz is installed and configured: {e}")
